In [1]:
# app.py
from time import perf_counter  # 新增：计时函数

from modeling import build_models_from_csv
from bundles import BaseBundle, MSBundle
from utils import (
    tighten_bounds_one_model,
    MIN_DIST, ACTIVE_TOL, GAP_STOP_TOL,
)
from simplex import run_pid_simplex_3d
from bundles import BaseBundle, MSBundle, QMinBundle


RUN_QUICK_TEST = True  # True: 先用小规模验证

if RUN_QUICK_TEST:
    csv_path       = "data.csv"
    max_scenarios  = 2
    target_nodes   = 13
else:
    csv_path       = "data.csv"
    max_scenarios  = 99
    target_nodes   = 30

bounds = {
    "x":  (None, None),
    "u":  (None, None),
    "e":  (None, None),
    "I":  (-10, 10),
    "Kp": (0, 1),
    "Ki": (0, 1),
    "Kd": (0, 1),
}
weights = (1.0, 0.01)

# ====== 阶段 1：数据加载与场景构造 ======
t_load0 = perf_counter()
model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)
t_load1 = perf_counter()
print(f"[Time] Data load & scenario build: {t_load1 - t_load0:.3f}s")

# ====== 阶段 1.5：FBBT / OBBT（在持久化实例化之前进行）======
# 对齐他们 PID 脚本的默认：FBBT 开、OBBT 开，OBBT 用轻量选项
obbt_solver_opts = {
    "NonConvex": 2,
    "MIPGap": 1,     # 宽松
    "TimeLimit": 5   # 轻量
}
for m, yvars in zip(model_list, first_stg_vars_list):
    tighten_bounds_one_model(m, yvars,
                             use_fbbt=True,
                             use_obbt=True,
                             obbt_solver_name="gurobi",
                             obbt_solver_opts=obbt_solver_opts,
                             max_rounds=3, tol=1e-6, verbose=True)

# ====== 阶段 2：求解器持久化包装 ======
ub_options = {
    'NonConvex': 2,           # 他们的 UB 只强调非凸
    # 按需可再加 TimeLimit 等；他们脚本里 UB 不怎么加别的
}
lb_options = {
    'NonConvex': 2,
    'MIPGap': 0.2,            # 他们 LB 初始较宽
    'TimeLimit': 15           # 他们 LB 设了时间上限
}

t_wrap0 = perf_counter()
base_bundles = [BaseBundle(m, ub_options) for m in model_list]  # UB 侧
ms_bundles   = [MSBundle(m, yvars, lb_options) for m, yvars in zip(model_list, first_stg_vars_list)]  # LB（ms）
qmin_bundles = [QMinBundle(m, yvars, lb_options) for m, yvars in zip(model_list, first_stg_vars_list)]  # 新增：真Q最小化

t_wrap1 = perf_counter()
print(f"[Time] Persistent wrapper (GurobiPersistent) setup: {t_wrap1 - t_wrap0:.3f}s")

# ====== stage 3：main loop ======
agg_bundle = None  # 对每个场景分别算，不启用共享 λ 聚合

t_run0 = perf_counter()
hist = run_pid_simplex_3d(
    base_bundles=base_bundles,
    ms_bundles=ms_bundles,
    qmin_bundles=qmin_bundles,          # ← 新增
    model_list=model_list,
    first_vars_list=first_stg_vars_list,
    target_nodes=target_nodes,
    min_dist=MIN_DIST,
    active_tol=ACTIVE_TOL,
    verbose=True,
    agg_bundle=agg_bundle,
    gap_stop_tol=1e-1,
)

t_run1 = perf_counter()
print(f"[Time] Main loop total: {t_run1 - t_run0:.3f}s")

# ====== 结果输出 ======
print("\n==== Done ====")
print(f"Total nodes: {len(hist['nodes'])}")
print(f"Best UB: {min(hist['UB_hist']) if hist['UB_hist'] else None}")
print(f"Last LB: {hist['LB_hist'][-1] if hist['LB_hist'] else None}")

[Time] Data load & scenario build: 0.012s
[Tighten] rounds=1, changed=False
[Tighten] rounds=1, changed=False
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter MIPGap to value 0.2
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 15
Set parameter MIPGap to value 0.2
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 15
Set parameter MIPGap to value 0.2
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 15
Set parameter MIPGap to value 0.2
Set parameter NumericFocus to value 1
Set p